In [ ]:
# Python 3.12で動作確認
# 最初に一度だけパッケージインストール
# %pip install msal==1.36.0 requests==2.33.1 pandas==3.0.2

In [ ]:
import json
import time
from numbers import Real
from typing import Sequence, Optional

import msal
import pandas as pd
import requests

# 1) 設定

https://learn.microsoft.com/ja-jp/rest/api/fabric/articles/get-started/create-entra-app

Fabric APIのためにEntraにアプリ登録を行う

In [ ]:
CLIENT_ID = "2b9a29b8-b584-4072-a810-b7b8bd7e9e36" # Entraに登録したアプリのクライアントID
AUTHORITY = "https://login.microsoftonline.com/organizations" # 参考: https://learn.microsoft.com/ja-jp/entra/identity-platform/msal-client-application-configuration

WORKSPACE_ID = "9176c11a-85af-4766-83da-3714c3878d20" # MLモデルエンドポイントのcurlコマンドから特定
MODEL_ID = "323af236-3675-4512-bc18-2f14d2b1413d" # MLモデルエンドポイントのcurlコマンドから特定

# score API なら Item.ReadWrite.All か MLModel.ReadWrite.All でOK
SCOPES = [
    "https://api.fabric.microsoft.com/Item.ReadWrite.All",
    # "https://api.fabric.microsoft.com/MLModel.ReadWrite.All",
]

BASE_URL = "https://api.fabric.microsoft.com/v1"

# 2) アクセストークン取得

別タブで認証ページが開く
十分なアクセス権を持つユーザーで認証する

ユーザー認証ではなく機械認証が必要な場合はサービスプリンシパルやマネージドIDの使用を検討
本サンプルはユーザー認証前提となっているため、アクセストークン取得周りの実装に修正が必要になる

https://learn.microsoft.com/ja-jp/rest/api/fabric/articles/identity-support

In [ ]:
def acquire_fabric_access_token(
    client_id: str,
    authority: str,
    scopes: Sequence[str],
) -> str:
    app = msal.PublicClientApplication(
        client_id=client_id,
        authority=authority,
    )

    result = None
    accounts = app.get_accounts()
    if accounts:
        result = app.acquire_token_silent(list(scopes), account=accounts[0])

    if not result:
        result = app.acquire_token_interactive(
            scopes=list(scopes),
            timeout=300,
        )

    if "access_token" not in result:
        raise RuntimeError(
            "Token acquisition failed:\n"
            + json.dumps(result, ensure_ascii=False, indent=2)
        )

    return result["access_token"]

# 3) 入力payload作成

In [ ]:
def build_score_payload(rows: Sequence[Sequence[float]]) -> dict:
    normalized_rows = []

    for row_idx, row in enumerate(rows, start=1):
        if len(row) != 5:
            raise ValueError(
                f"Row {row_idx} must contain exactly 5 values, got {len(row)}."
            )

        normalized_row = []
        for col_idx, value in enumerate(row, start=1):
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError(
                    f"Row {row_idx}, column {col_idx} must be a numeric value, got {type(value).__name__}."
                )
            normalized_row.append(float(value))

        normalized_rows.append(normalized_row)

    return {
        "formatType": "dataframe",
        "orientation": "values",
        "inputs": normalized_rows,
    }

# 4) score API 呼び出し

In [ ]:
def score_fabric_model(
    access_token: str,
    workspace_id: str,
    model_id: str,
    rows: Sequence[Sequence[float]],
    version_name: Optional[str] = None,
    request_timeout_sec: int = 60,
    operation_timeout_sec: int = 600,
):
    payload = build_score_payload(rows)

    if version_name:
        url = (
            f"{BASE_URL}/workspaces/{workspace_id}"
            f"/mlModels/{model_id}/endpoint/versions/{version_name}/score"
        )
    else:
        url = (
            f"{BASE_URL}/workspaces/{workspace_id}"
            f"/mlModels/{model_id}/endpoint/score"
        )

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    with requests.Session() as session:
        response = session.post(
            url,
            headers=headers,
            json=payload,
            timeout=request_timeout_sec,
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 202:
            poll_url = response.headers.get("Location")
            if not poll_url:
                return {
                    "status_code": 202,
                    "message": "Accepted, but no Location header was returned.",
                    "headers": dict(response.headers),
                    "raw_text": response.text,
                }

            deadline = time.time() + operation_timeout_sec
            retry_after = int(response.headers.get("Retry-After", "10"))

            while time.time() < deadline:
                time.sleep(retry_after)

                poll_response = session.get(
                    poll_url,
                    headers={"Authorization": f"Bearer {access_token}"},
                    timeout=request_timeout_sec,
                )

                if poll_response.status_code == 202:
                    retry_after = int(poll_response.headers.get("Retry-After", "10"))
                    continue

                if poll_response.ok:
                    try:
                        return poll_response.json()
                    except ValueError:
                        return {
                            "status_code": poll_response.status_code,
                            "raw_text": poll_response.text,
                        }

                raise requests.HTTPError(
                    f"Polling failed: {poll_response.status_code}\n{poll_response.text}",
                    response=poll_response,
                )

            raise TimeoutError(
                f"Operation did not complete within {operation_timeout_sec} seconds."
            )

        try:
            error_body = response.json()
        except ValueError:
            error_body = response.text

        raise requests.HTTPError(
            f"Request failed: {response.status_code}\n{error_body}",
            response=response,
        )

# 5) 実行例

In [ ]:
sample_rows = [
    [0.12, -0.34, 1.23, 4.56, -7.89],
    [1.00, 2.00, 3.00, 4.00, 5.00],
]

access_token = acquire_fabric_access_token(
    client_id=CLIENT_ID,
    authority=AUTHORITY,
    scopes=SCOPES,
)

result = score_fabric_model(
    access_token=access_token,
    workspace_id=WORKSPACE_ID,
    model_id=MODEL_ID,
    rows=sample_rows,
    # version_name="1",   # 特定バージョンを叩くなら設定
)

print(json.dumps(result, ensure_ascii=False, indent=2))

if isinstance(result, dict) and "predictions" in result:
    predictions_df = pd.DataFrame(result["predictions"])
    display(predictions_df)